# Day 13 — Tune, register, deploy

PIT features from Day 12. Label `IS_FULFILLED` (registered as lab name `churn_model`).
Warehouse `LEARN_WH`. Enable notebook packages: `scikit-learn`, `pandas`.


## 1. GridSearchCV

Sweep `n_estimators`, `max_depth`, `min_samples_leaf`. Scoring = **F1**.
After `fit`, call `to_sklearn()` for `best_params_` / `best_score_`.


In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.modeling.model_selection import GridSearchCV
from snowflake.ml.modeling.metrics import f1_score as sp_f1
from snowflake.ml.registry import Registry
from snowflake.ml.model import task as ml_task

session = get_active_session()
feat_cols = ["LOG_PRIOR_SPEND", "DAYS_SINCE_FIRST_ORDER", "PRIORITY_ORDINAL"]
pit = session.table("RETAIL_LAKEHOUSE.FEATURE_STORE.PIT_TRAINING_SET")
train_df, test_df = pit.random_split([0.8, 0.2], seed=42)

gs = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid={
        "n_estimators": [10, 20, 40],
        "max_depth": [4, 6, 8],
        "min_samples_leaf": [1, 4],
    },
    scoring="f1",
    cv=3,
    input_cols=feat_cols,
    label_cols=["IS_FULFILLED"],
    output_cols=["PREDICTED"],
)
gs.fit(train_df)
sk = gs.to_sklearn()
print("best_params_", sk.best_params_)
print("best_score_", sk.best_score_)


## 2. Metric

Use **F1** (and report AUC). Classes are ~50/50; missing a fulfilled order and
false-alarming an open one both cost operations. Accuracy alone is a coin flip
on Day 11 ad-hoc columns (0.515). Live tuned test: **F1 0.959 · AUC 0.990 ·
acc 0.958**. `best_params_`: `max_depth=8`, `min_samples_leaf=4`, `n_estimators=10`.


## 3–4. Register `churn_model` version `v1` and set champion

Snowflake stores identifiers as `CHURN_MODEL` / `V1`. Downstream uses `.default`
so scoring code never hard-codes a version.


In [ ]:
session.sql("CREATE SCHEMA IF NOT EXISTS RETAIL_LAKEHOUSE.ML_MODELS").collect()
reg = Registry(session=session, database_name="RETAIL_LAKEHOUSE", schema_name="ML_MODELS")

sample = train_df.select(*feat_cols).limit(20)
mv = reg.log_model(
    gs.to_sklearn().best_estimator_,
    model_name="churn_model",
    version_name="v1",
    conda_dependencies=["scikit-learn"],
    comment="Day 13 RF on PIT customer features",
    metrics={"f1": 0.9591, "auc": 0.9902, "cv_f1": 0.9501},
    sample_input_data=sample,
    task=ml_task.Task.TABULAR_BINARY_CLASSIFICATION,
)
model = reg.get_model("churn_model")
model.default = "v1"
mv.set_alias("PRODUCTION")
print("default", model.default.version_name)
print(mv.show_functions())


## 5. Batch inference

`reg.get_model('churn_model').default.run(..., function_name='predict')` scores
the full PIT table into `GOLD.CHURN_SCORES` (3,926 rows).


In [ ]:
score_df = session.table("RETAIL_LAKEHOUSE.FEATURE_STORE.PIT_TRAINING_SET")
preds = reg.get_model("churn_model").default.run(score_df, function_name="predict")
preds.write.mode("overwrite").save_as_table("RETAIL_LAKEHOUSE.GOLD.CHURN_SCORES")
session.table("RETAIL_LAKEHOUSE.GOLD.CHURN_SCORES").limit(8).show()
print("rows", session.table("RETAIL_LAKEHOUSE.GOLD.CHURN_SCORES").count())


## 6. Real-time SQL

Trial allows `CHURN_MODEL!PREDICT(...)`. One test row (customer 385, day 7,
priority ordinal 2) returned `{"output_feature_0": 1}` — same as the batch table.

```sql
USE SCHEMA RETAIL_LAKEHOUSE.ML_MODELS;
SELECT CHURN_MODEL!PREDICT(12.110107796476692::FLOAT, 7::NUMBER, 2.0::FLOAT);
```
